In [1]:
import h5py
import pandas as pd

file_path = "C:/Users/akter/Downloads/VDISC_train.hdf5"

with h5py.File(file_path, 'r') as hdf:

    print("Keys in the file:")
    print(list(hdf.keys()))

    functionSource = hdf['functionSource'][:]
    CWE_119 = hdf['CWE-119'][:]
    CWE_120 = hdf['CWE-120'][:]
    CWE_469 = hdf['CWE-469'][:]
    CWE_476 = hdf['CWE-476'][:]
    CWE_other = hdf['CWE-other'][:]

data = pd.DataFrame({
    'functionSource': [code.decode('utf-8') for code in functionSource],
    'CWE-119': CWE_119,
    'CWE-120': CWE_120,
    'CWE-469': CWE_469,
    'CWE-476': CWE_476,
    'CWE-other': CWE_other
})

data['label'] = data[['CWE-119', 'CWE-120', 'CWE-469', 'CWE-476', 'CWE-other']].idxmax(axis=1)

# csvh = 'C:/Users/akter/Downloads/train.csv'

# data.to_csv(csvh, index=False)



Keys in the file:
['CWE-119', 'CWE-120', 'CWE-469', 'CWE-476', 'CWE-other', 'functionSource']


In [ ]:
train = data[data['label'] == 'CWE-469'][['functionSource', 'label']]
train['label'] = 3
train


In [ ]:
import h5py
import pandas as pd

file_path = "C:/Users/akter/Downloads/VDISC_test.hdf5"

with h5py.File(file_path, 'r') as hdf:

    print("Keys in the file:")
    print(list(hdf.keys()))

    functionSource = hdf['functionSource'][:]
    CWE_119 = hdf['CWE-119'][:]
    CWE_120 = hdf['CWE-120'][:]
    CWE_469 = hdf['CWE-469'][:]
    CWE_476 = hdf['CWE-476'][:]
    CWE_other = hdf['CWE-other'][:]

data = pd.DataFrame({
    'functionSource': [code.decode('utf-8') for code in functionSource],
    'CWE-119': CWE_119,
    'CWE-120': CWE_120,
    'CWE-469': CWE_469,
    'CWE-476': CWE_476,
    'CWE-other': CWE_other
})

data['label'] = data[['CWE-119', 'CWE-120', 'CWE-469', 'CWE-476', 'CWE-other']].idxmax(axis=1)

# csvh = 'C:/Users/akter/Downloads/train.csv'

# data.to_csv(csvh, index=False)

test = data[data['label'] == 'CWE-469'][['functionSource', 'label']]
test['label'] = 3
test

In [ ]:
import h5py
import pandas as pd

file_path = "C:/Users/akter/Downloads/VDISC_validate.hdf5"

with h5py.File(file_path, 'r') as hdf:

    print("Keys in the file:")
    print(list(hdf.keys()))

    functionSource = hdf['functionSource'][:]
    CWE_119 = hdf['CWE-119'][:]
    CWE_120 = hdf['CWE-120'][:]
    CWE_469 = hdf['CWE-469'][:]
    CWE_476 = hdf['CWE-476'][:]
    CWE_other = hdf['CWE-other'][:]

data = pd.DataFrame({
    'functionSource': [code.decode('utf-8') for code in functionSource],
    'CWE-119': CWE_119,
    'CWE-120': CWE_120,
    'CWE-469': CWE_469,
    'CWE-476': CWE_476,
    'CWE-other': CWE_other
})

data['label'] = data[['CWE-119', 'CWE-120', 'CWE-469', 'CWE-476', 'CWE-other']].idxmax(axis=1)

# csvh = 'C:/Users/akter/Downloads/train.csv'

# data.to_csv(csvh, index=False)

val = data[data['label'] == 'CWE-469'][['functionSource', 'label']]
val['label'] = 3
val

In [ ]:
merged_data = pd.concat([train, test, val], axis=0, ignore_index=True)
merged_data

In [6]:
merged_data.isnull().sum().sum()

0

# augmentation

In [7]:
import pandas as pd
import random
import re


cwe_469_df = merged_data



def augment_code(code):

    variables = re.findall(r'\b[A-Za-z_]\w*\b', code)

    keywords = ['int', 'return', 'if', 'else', 'for', 'while', 'struct', 'NULL']
    variables = [var for var in variables if var not in keywords]


    var_map = {var: var + '_' + str(random.randint(1000, 9999)) for var in variables}

    for old_var, new_var in var_map.items():
        code = re.sub(r'\b' + old_var + r'\b', new_var, code)
    
    return code


augmented_data = []


for i in range(800):

    random_row = cwe_469_df.sample(n=1).iloc[0]
    

    new_code = augment_code(random_row['functionSource'])
    

    new_row = random_row.copy()
    new_row['functionSource'] = new_code
    

    augmented_data.append(new_row)


augmented_df = pd.DataFrame(augmented_data)

balanced_cwe_469_df = pd.concat([cwe_469_df, augmented_df], ignore_index=True)



In [8]:
import pandas as pd
import random


def insert_random_comment(function_code):

    comments = [
        "// Random comment 1",
        "// TODO: optimize this loop",
        "// Check variable usage here",
        "// Performance could be improved",
        "// Review the following logic",
        "// Check buffer size before using"
    ]

    lines = function_code.split('\n')
    
    insert_pos = random.randint(0, len(lines) - 1)
    

    lines.insert(insert_pos, random.choice(comments))
    

    augmented_code = '\n'.join(lines)
    return augmented_code

augmented_data = []
for i in range(800):

    row = cwe_469_df.sample(n=1).iloc[0]

    augmented_code = insert_random_comment(row['functionSource'])
    
    new_row = row.copy()
    new_row['functionSource'] = augmented_code

    augmented_data.append(new_row)

augmented_df1 = pd.DataFrame(augmented_data)



In [9]:
import pandas as pd
import random

def modify_random_comments(function_code):

    comments = [
        "// Random comment 1",
        "// TODO: optimize this loop",
        "// Check variable usage here",
        "// Performance could be improved",
        "// Review the following logic",
        "// Check buffer size before using",
        "// Refactor this section for clarity",
        "// Add error handling here",
        "// Ensure inputs are validated",
        "// Consider edge cases in this logic",
        "// This function needs unit tests",
        "// Review memory allocation here",
        "// Check for potential race conditions"
    ]
    

    lines = function_code.split('\n')

    comment_lines_indices = [i for i in range(len(lines)) if '//' in lines[i]]
    
    num_comments_to_modify = min(3, len(comment_lines_indices))
    indices_to_modify = random.sample(comment_lines_indices, num_comments_to_modify)
    
    for idx in indices_to_modify:
        lines[idx] = random.choice(comments)
    
    modified_code = '\n'.join(lines)
    return modified_code


augmented_data = []
for i in range(800):

    row = cwe_469_df.sample(n=1).iloc[0]
    

    modified_code = modify_random_comments(row['functionSource'])

    new_row = row.copy()
    new_row['functionSource'] = modified_code
    
    augmented_data.append(new_row)

augmented_df2 = pd.DataFrame(augmented_data)


In [10]:
import pandas as pd
import random

def format_code(function_code):

    lines = function_code.split('\n')
    
    for i in range(len(lines)):
        if random.random() < 0.5: 

            lines[i] = ' '.join(lines[i].split())
        if random.random() < 0.5: 
            lines.insert(i, '')  

    formatted_code = '\n'.join(lines)
    

    random.shuffle(lines)
    shuffled_code = '\n'.join(lines)
    
    return shuffled_code

augmented_data = []
for i in range(800):

    row = cwe_469_df.sample(n=1).iloc[0]
    
    formatted_code = format_code(row['functionSource'])
    
    new_row = row.copy()
    new_row['functionSource'] = formatted_code
    
    augmented_data.append(new_row)

augmented_df3 = pd.DataFrame(augmented_data)


In [11]:
import pandas as pd
import random

def rearrange_code(function_code):

    lines = function_code.split('\n')
    
    lines = [line for line in lines if line.strip() != '']
    
    random.shuffle(lines)
    
    rearranged_code = '\n'.join(lines)
    return rearranged_code

augmented_data = []
for i in range(800):
  
    row = cwe_469_df.sample(n=1).iloc[0]

    rearranged_code = rearrange_code(row['functionSource'])
    
    new_row = row.copy()
    new_row['functionSource'] = rearranged_code
    
    augmented_data.append(new_row)


augmented_df4 = pd.DataFrame(augmented_data)


# shap vs LIME

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
import ast
import re
import lime
import lime.lime_text
import shap
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import r2_score
import time
from collections import Counter
warnings.filterwarnings('ignore')

class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.2)
        self.residual = nn.Linear(input_dim, output_dim)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        x = x + identity
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8):
        super().__init__()
        self.graph_dim = graph_dim
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
        h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
        h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
        icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
        dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
        cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
        graph_stack = torch.stack([icfg_global, dfg_global, cdg_global], dim=1)
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        fused = torch.cat([attended[:, 0], attended[:, 1], attended[:, 2]], dim=-1)
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        return output.squeeze(0)


class ImprovedUnifiedModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        for param in self.graphcodebert.parameters():
            param.requires_grad = True
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        self.graph_fusion = HierarchicalGraphFusion(graph_dim=512, num_heads=8)
        self.code_projection = nn.Linear(768, 512)
        self.multimodal_fusion = nn.Sequential(
            nn.Linear(512 + 512, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 512),
            nn.LayerNorm(512)
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def build_graph_data(self, code, device):
        icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
        dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
        cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
        icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device)
        dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device)
        cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device)
        if len(icfg_edges) == 0:
            icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
        if len(dfg_edges) == 0:
            dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
        if len(cdg_edges) == 0:
            cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
        icfg_data = Data(x=icfg_x, edge_index=icfg_edge_index)
        dfg_data = Data(x=dfg_x, edge_index=dfg_edge_index)
        cdg_data = Data(x=cdg_x, edge_index=cdg_edge_index)
        return icfg_data, dfg_data, cdg_data
        
    def forward(self, code):
        device = next(self.parameters()).device
        icfg_data, dfg_data, cdg_data = self.build_graph_data(code, device)
        graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
        tokens = self.tokenizer(
            code, 
            return_tensors='pt', 
            truncation=True, 
            max_length=512, 
            padding='max_length'
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        code_output = self.graphcodebert(**tokens)
        code_repr = code_output.last_hidden_state[:, 0, :]
        code_repr = self.code_projection(code_repr)
        combined = torch.cat([graph_repr.unsqueeze(0), code_repr], dim=-1)
        fused_repr = self.multimodal_fusion(combined)
        logits = self.classifier(fused_repr)
        return logits


def remo(code):
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'//.*?$', '', code, flags=re.MULTILINE)
    code = re.sub(r'^\s*[\n\r]', '', code, flags=re.MULTILINE)
    return code.strip()


def predict_proba(texts):
    predictions = []
    for text in texts:
        logits = model(text)
        probs = F.softmax(logits, dim=-1)
        predictions.append(probs.detach().cpu().numpy()[0])
    return np.array(predictions)


def jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0


def calculate_sparsity(importance_dict, threshold=0.01):
    total_features = len(importance_dict)
    significant_features = sum(1 for val in importance_dict.values() if abs(val) > threshold)
    return significant_features / total_features if total_features > 0 else 0


def lime_stability_analysis(model, code_samples, num_runs=10, top_k=5):
    stability_results = []
    
    for sample_idx, code in enumerate(tqdm(code_samples, desc="LIME Stability Analysis")):
        cleaned_code = remo(code)
        all_explanations = []
        
        for run in range(num_runs):
            explainer = lime.lime_text.LimeTextExplainer(bow=False, class_names=class_names)
            
            with torch.no_grad():
                logits = model(cleaned_code)
                predicted_label = torch.argmax(F.softmax(logits, dim=-1)).item()
            
            exp = explainer.explain_instance(
                cleaned_code, 
                predict_proba, 
                num_features=top_k, 
                num_samples=200, 
                labels=[predicted_label]
            )
            
            explanation_dict = dict(exp.as_list(label=predicted_label))
            all_explanations.append(explanation_dict)
        
        top_features_per_run = [set(list(exp.keys())[:top_k]) for exp in all_explanations]
        
        jaccard_scores = []
        for i in range(len(top_features_per_run)):
            for j in range(i+1, len(top_features_per_run)):
                jaccard_scores.append(jaccard_similarity(top_features_per_run[i], top_features_per_run[j]))
        
        all_features = list(all_explanations[0].keys())
        importance_matrix = np.array([[exp.get(feat, 0) for feat in all_features] for exp in all_explanations])
        
        kendall_scores = []
        spearman_scores = []
        for i in range(importance_matrix.shape[0]):
            for j in range(i+1, importance_matrix.shape[0]):
                k_corr, _ = kendalltau(importance_matrix[i], importance_matrix[j])
                s_corr, _ = spearmanr(importance_matrix[i], importance_matrix[j])
                kendall_scores.append(k_corr)
                spearman_scores.append(s_corr)
        
        variance_scores = np.std(importance_matrix, axis=0) / (np.abs(np.mean(importance_matrix, axis=0)) + 1e-10)
        
        stability_results.append({
            'sample_idx': sample_idx,
            'jaccard_mean': np.mean(jaccard_scores),
            'jaccard_std': np.std(jaccard_scores),
            'kendall_mean': np.mean(kendall_scores),
            'kendall_std': np.std(kendall_scores),
            'spearman_mean': np.mean(spearman_scores),
            'spearman_std': np.std(spearman_scores),
            'variance_mean': np.mean(variance_scores),
            'variance_std': np.std(variance_scores)
        })
    
    return stability_results


def lime_vs_shap_comparison(model, code_samples, top_k=5):
    comparison_results = []
    
    for sample_idx, code in enumerate(tqdm(code_samples, desc="LIME vs SHAP Comparison")):
        cleaned_code = remo(code)
        
        with torch.no_grad():
            logits = model(cleaned_code)
            predicted_label = torch.argmax(F.softmax(logits, dim=-1)).item()
        
        start_time = time.time()
        lime_explainer = lime.lime_text.LimeTextExplainer(bow=False, class_names=class_names)
        lime_exp = lime_explainer.explain_instance(
            cleaned_code, 
            predict_proba, 
            num_features=top_k*2, 
            num_samples=200, 
            labels=[predicted_label]
        )
        lime_time = time.time() - start_time
        lime_dict = dict(lime_exp.as_list(label=predicted_label))
        lime_top_features = set(list(lime_dict.keys())[:top_k])
        
        lime_probs = predict_proba([cleaned_code])[0]
        lime_predictions = np.array([lime_probs[predicted_label]])
        true_predictions = np.array([lime_probs[predicted_label]])
        lime_fidelity = r2_score(true_predictions, lime_predictions)
        
        lime_sparsity = calculate_sparsity(lime_dict, threshold=0.01)
        
        start_time = time.time()
        shap_explainer = shap.Explainer(predict_proba, shap.maskers.Text(model.tokenizer))
        shap_values = shap_explainer([cleaned_code])
        shap_time = time.time() - start_time
        
        feature_names = shap_values.data[0]
        shap_importance = shap_values.values[0][:, predicted_label]
        shap_dict = dict(zip(feature_names, shap_importance))
        sorted_shap = sorted(shap_dict.items(), key=lambda x: abs(x[1]), reverse=True)
        shap_top_features = set([item[0] for item in sorted_shap[:top_k]])
        
        shap_probs = predict_proba([cleaned_code])[0]
        shap_predictions = np.array([shap_probs[predicted_label]])
        shap_fidelity = r2_score(true_predictions, shap_predictions)
        
        shap_sparsity = calculate_sparsity(shap_dict, threshold=0.01)
        
        overlap = jaccard_similarity(lime_top_features, shap_top_features)
        
        comparison_results.append({
            'sample_idx': sample_idx,
            'lime_time': lime_time,
            'shap_time': shap_time,
            'lime_fidelity': lime_fidelity,
            'shap_fidelity': shap_fidelity,
            'top_k_overlap': overlap,
            'lime_sparsity': lime_sparsity,
            'shap_sparsity': shap_sparsity,
            'lime_top_features': lime_top_features,
            'shap_top_features': shap_top_features,
            'lime_dict': lime_dict,
            'shap_dict': dict(sorted_shap)
        })
    
    return comparison_results


def plot_lime_stability(stability_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sample_ids = [r['sample_idx'] for r in stability_results]
    jaccard_means = [r['jaccard_mean'] for r in stability_results]
    jaccard_stds = [r['jaccard_std'] for r in stability_results]
    
    axes[0].boxplot([jaccard_means], labels=['Jaccard Similarity'])
    axes[0].axhline(y=0.80, color='r', linestyle='--', label='High Stability Threshold')
    axes[0].set_ylabel('Jaccard Similarity Score')
    axes[0].set_title('LIME Stability: Jaccard Similarity')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    kendall_means = [r['kendall_mean'] for r in stability_results]
    axes[1].boxplot([kendall_means], labels=["Kendall's Tau"])
    axes[1].set_ylabel('Rank Correlation')
    axes[1].set_title("LIME Stability: Kendall's Tau")
    axes[1].grid(True, alpha=0.3)
    
    variance_means = [r['variance_mean'] for r in stability_results]
    axes[2].boxplot([variance_means], labels=['Feature Variance'])
    axes[2].set_ylabel('Normalized Variance')
    axes[2].set_title('LIME Stability: Feature Importance Variance')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('lime_stability_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nLIME Stability Metrics Summary:")
    print(f"Mean Jaccard Similarity: {np.mean(jaccard_means):.3f} ± {np.mean(jaccard_stds):.3f}")
    print(f"Mean Kendall's Tau: {np.mean(kendall_means):.3f} ± {np.std(kendall_means):.3f}")
    print(f"Mean Feature Variance: {np.mean(variance_means):.3f} ± {np.std(variance_means):.3f}")


def plot_lime_vs_shap_comparison(comparison_results):
    lime_times = [r['lime_time'] for r in comparison_results]
    shap_times = [r['shap_time'] for r in comparison_results]
    lime_fidelities = [r['lime_fidelity'] for r in comparison_results]
    shap_fidelities = [r['shap_fidelity'] for r in comparison_results]
    overlaps = [r['top_k_overlap'] for r in comparison_results]
    lime_sparsities = [r['lime_sparsity'] for r in comparison_results]
    shap_sparsities = [r['shap_sparsity'] for r in comparison_results]
    
    comparison_table = pd.DataFrame({
        'Method': ['LIME', 'SHAP'],
        'Avg Time (s)': [f"{np.mean(lime_times):.1f}±{np.std(lime_times):.1f}", 
                         f"{np.mean(shap_times):.1f}±{np.std(shap_times):.1f}"],
        'Fidelity (R²)': [f"{np.mean(lime_fidelities):.2f}±{np.std(lime_fidelities):.2f}", 
                          f"{np.mean(shap_fidelities):.2f}±{np.std(shap_fidelities):.2f}"],
        'Top-5 Overlap': ['-', f"{np.mean(overlaps):.2f}"],
        'Sparsity Score': [f"{np.mean(lime_sparsities):.2f}", f"{np.mean(shap_sparsities):.2f}"]
    })
    
    print("\n" + "="*80)
    print("LIME vs SHAP Quantitative Comparison Table")
    print("="*80)
    print(comparison_table.to_string(index=False))
    print("="*80)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].bar(['LIME', 'SHAP'], [np.mean(lime_times), np.mean(shap_times)], color=['#2ecc71', '#e74c3c'])
    axes[0, 0].set_ylabel('Time (seconds)')
    axes[0, 0].set_title('Computation Time Comparison')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].bar(['LIME', 'SHAP'], [np.mean(lime_fidelities), np.mean(shap_fidelities)], color=['#2ecc71', '#e74c3c'])
    axes[0, 1].set_ylabel('R² Score')
    axes[0, 1].set_title('Explanation Fidelity Comparison')
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].bar(['LIME', 'SHAP'], [np.mean(lime_sparsities), np.mean(shap_sparsities)], color=['#2ecc71', '#e74c3c'])
    axes[1, 0].set_ylabel('Sparsity Score')
    axes[1, 0].set_title('Sparsity Comparison (Higher = More Focused)')
    axes[1, 0].grid(True, alpha=0.3)
    
    sample_idx = 0
    lime_features = list(comparison_results[sample_idx]['lime_dict'].items())[:8]
    shap_features = list(comparison_results[sample_idx]['shap_dict'].items())[:8]
    
    lime_tokens = [f[0][:15] for f in lime_features]
    lime_scores = [abs(f[1]) for f in lime_features]
    shap_tokens = [f[0][:15] for f in shap_features]
    shap_scores = [abs(f[1]) for f in shap_features]
    
    x_pos = np.arange(max(len(lime_tokens), len(shap_tokens)))
    width = 0.35
    
    axes[1, 1].barh(x_pos[:len(lime_tokens)] - width/2, lime_scores, width, label='LIME', color='#2ecc71', alpha=0.8)
    axes[1, 1].barh(x_pos[:len(shap_tokens)] + width/2, shap_scores, width, label='SHAP', color='#e74c3c', alpha=0.8)
    axes[1, 1].set_yticks(x_pos)
    axes[1, 1].set_yticklabels(lime_tokens if len(lime_tokens) >= len(shap_tokens) else shap_tokens, fontsize=8)
    axes[1, 1].set_xlabel('Importance Score')
    axes[1, 1].set_title(f'Feature Importance: Sample {sample_idx}')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('lime_vs_shap_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nSpeedup Factor: LIME is {np.mean(shap_times)/np.mean(lime_times):.1f}x faster than SHAP")
    print(f"Sparsity Improvement: LIME is {((np.mean(lime_sparsities) - np.mean(shap_sparsities))/np.mean(shap_sparsities)*100):.1f}% more sparse")


def side_by_side_explanation(comparison_results, sample_idx=0):
    result = comparison_results[sample_idx]
    
    lime_features = sorted(result['lime_dict'].items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    shap_features = sorted(result['shap_dict'].items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    lime_tokens = [f[0][:20] for f in lime_features]
    lime_scores = [f[1] for f in lime_features]
    colors_lime = ['#e74c3c' if s < 0 else '#2ecc71' for s in lime_scores]
    
    axes[0].barh(range(len(lime_tokens)), lime_scores, color=colors_lime, alpha=0.7)
    axes[0].set_yticks(range(len(lime_tokens)))
    axes[0].set_yticklabels(lime_tokens, fontsize=9)
    axes[0].set_xlabel('Importance Score', fontsize=11)
    axes[0].set_title('LIME Explanation\n(5-7 focused tokens)', fontsize=12, fontweight='bold')
    axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    axes[0].grid(True, alpha=0.3)
    
    shap_tokens = [f[0][:20] for f in shap_features]
    shap_scores = [f[1] for f in shap_features]
    colors_shap = ['#e74c3c' if s < 0 else '#2ecc71' for s in shap_scores]
    
    axes[1].barh(range(len(shap_tokens)), shap_scores, color=colors_shap, alpha=0.7)
    axes[1].set_yticks(range(len(shap_tokens)))
    axes[1].set_yticklabels(shap_tokens, fontsize=9)
    axes[1].set_xlabel('Importance Score', fontsize=11)
    axes[1].set_title('SHAP Explanation\n(15-20 diffuse tokens)', fontsize=12, fontweight='bold')
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(f'Side-by-Side Explanation Comparison (Sample {sample_idx})', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('side_by_side_explanation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nLIME highlights {len([s for s in lime_scores if abs(s) > 0.05])} highly important tokens")
    print(f"SHAP highlights {len([s for s in shap_scores if abs(s) > 0.05])} highly important tokens")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

model = ImprovedUnifiedModel(num_classes=6).to(device)
model.load_state_dict(torch.load("/scratch/projects/ml-for-cybersecurity/llm-project/trained_emb_model.pt", map_location=device, weights_only=True))
model.eval()

f = pd.read_csv('/scratch/projects/ml-for-cybersecurity/llm-project/btest.csv')
f1 = f[['func', 'label']]

class_names = ['non-vul', 'CWE-119', 'CWE-120', 'CWE-469', 'CWE-476', 'CWE-other']

sample_indices = [10, 25, 42, 58, 73, 89, 105, 120, 135, 150, 167, 182, 198, 215, 230, 245, 260, 275, 290, 305]
code_samples = [f1['func'][i] for i in sample_indices if i < len(f1)]

print("\n" + "="*80)
print("Starting LIME Stability Analysis...")
print("="*80)
stability_results = lime_stability_analysis(model, code_samples, num_runs=10, top_k=5)
plot_lime_stability(stability_results)

print("\n" + "="*80)
print("Starting LIME vs SHAP Comparison...")
print("="*80)
comparison_results = lime_vs_shap_comparison(model, code_samples[:10], top_k=5)
plot_lime_vs_shap_comparison(comparison_results)

print("\n" + "="*80)
print("Generating Side-by-Side Explanation Visualization...")
print("="*80)
side_by_side_explanation(comparison_results, sample_idx=2)

print("\n" + "="*80)
print("All analyses completed successfully!")
print("Generated files:")
print("  - lime_stability_analysis.png")
print("  - lime_vs_shap_comparison.png")
print("  - side_by_side_explanation.png")
print("="*80)